# LLM as API Notebook — Gemini + System Prompt Endpoints + Tool Calling + SQLite

الفكرة:
- كل الـ endpoints المعقدة متعرفة داخل `SYSTEM_PROMPT`.
- المستخدم يكلم الشات طبيعي.
- Gemini يختار endpoint ويرجع JSON API response.
- عند الحاجة، Gemini يعمل Function Calling لأدوات قاعدة البيانات.
- قاعدة البيانات هنا SQLite محلية عشان النوتبوك يشتغل بسرعة.
- يوجد daily test suite يختبر endpoints يوميًا أو يدويًا داخل النوتبوك.

> قبل التشغيل: ضع مفتاح Gemini API في متغير بيئة باسم `GEMINI_API_KEY`.

In [ ]:
# تثبيت المكتبات
# شغل الخلية دي مرة واحدة فقط في Colab/Jupyter
!pip install -q -U google-genai pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.6.0 which is incompatible.


In [ ]:
import os
import json
import sqlite3
import uuid
from datetime import datetime, date
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field, ValidationError

try:
    from google import genai
    from google.genai import types
except Exception as e:
    genai = None
    types = None
    print("google-genai is not installed yet. Run the install cell first.")

## 1) إعدادات عامة

In [ ]:
DB_PATH = "prompt_api_runtime.sqlite3"
MODEL_NAME = "gemini-2.5-flash"

def now_iso() -> str:
    return datetime.utcnow().isoformat() + "Z"

def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"

def pretty(obj):
    print(json.dumps(obj, ensure_ascii=False, indent=2))

## 2) قاعدة البيانات المحلية

الجداول:
- `api_records`: تخزين عام لأي record من أي endpoint.
- `audit_logs`: لوج كامل لكل طلب.
- `daily_tests`: نتائج الاختبارات اليومية.

In [ ]:
def get_conn():
    return sqlite3.connect(DB_PATH)

def init_db():
    conn = get_conn()
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS api_records (
        id TEXT PRIMARY KEY,
        record_type TEXT NOT NULL,
        payload_json TEXT NOT NULL,
        created_at TEXT NOT NULL,
        updated_at TEXT NOT NULL
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS audit_logs (
        id TEXT PRIMARY KEY,
        user_message TEXT NOT NULL,
        endpoint TEXT,
        status_code INTEGER,
        request_json TEXT,
        response_json TEXT,
        created_at TEXT NOT NULL
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS daily_tests (
        id TEXT PRIMARY KEY,
        test_name TEXT NOT NULL,
        passed INTEGER NOT NULL,
        endpoint TEXT,
        response_json TEXT,
        created_at TEXT NOT NULL
    )
    """)

    conn.commit()
    conn.close()

init_db()
print("Database initialized:", DB_PATH)

Database initialized: prompt_api_runtime.sqlite3


## 3) Tools الخاصة بقاعدة البيانات

Gemini سيقدر يطلب استدعاء هذه الأدوات:
- `db_insert_record`
- `db_get_record`
- `db_search_records`
- `db_update_record`
- `db_delete_record`

مهم: الـ LLM لا يتصل مباشرة بالداتا بيز. هو فقط يطلب function call، والكود هنا هو الذي ينفذ.

In [ ]:
def db_insert_record(record_type: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    record_id = payload.get("id") or new_id(record_type)
    payload = dict(payload)
    payload["id"] = record_id

    ts = now_iso()
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        INSERT OR REPLACE INTO api_records (id, record_type, payload_json, created_at, updated_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        (record_id, record_type, json.dumps(payload, ensure_ascii=False), ts, ts)
    )
    conn.commit()
    conn.close()

    return {
        "status_code": 201,
        "message": "Record saved",
        "data": {
            "id": record_id,
            "record_type": record_type,
            "payload": payload
        },
        "errors": []
    }


def db_get_record(record_id: str) -> Dict[str, Any]:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT id, record_type, payload_json, created_at, updated_at FROM api_records WHERE id = ?", (record_id,))
    row = cur.fetchone()
    conn.close()

    if not row:
        return {
            "status_code": 404,
            "message": "Record not found",
            "data": None,
            "errors": [{"field": "record_id", "message": "No record exists with this id"}]
        }

    return {
        "status_code": 200,
        "message": "Record found",
        "data": {
            "id": row[0],
            "record_type": row[1],
            "payload": json.loads(row[2]),
            "created_at": row[3],
            "updated_at": row[4]
        },
        "errors": []
    }


def db_search_records(record_type: Optional[str] = None, query: Optional[str] = None, limit: int = 20) -> Dict[str, Any]:
    limit = min(max(int(limit), 1), 100)
    conn = get_conn()
    cur = conn.cursor()

    if record_type and query:
        cur.execute(
            """
            SELECT id, record_type, payload_json, created_at, updated_at
            FROM api_records
            WHERE record_type = ? AND payload_json LIKE ?
            ORDER BY updated_at DESC
            LIMIT ?
            """,
            (record_type, f"%{query}%", limit)
        )
    elif record_type:
        cur.execute(
            """
            SELECT id, record_type, payload_json, created_at, updated_at
            FROM api_records
            WHERE record_type = ?
            ORDER BY updated_at DESC
            LIMIT ?
            """,
            (record_type, limit)
        )
    elif query:
        cur.execute(
            """
            SELECT id, record_type, payload_json, created_at, updated_at
            FROM api_records
            WHERE payload_json LIKE ?
            ORDER BY updated_at DESC
            LIMIT ?
            """,
            (f"%{query}%", limit)
        )
    else:
        cur.execute(
            """
            SELECT id, record_type, payload_json, created_at, updated_at
            FROM api_records
            ORDER BY updated_at DESC
            LIMIT ?
            """,
            (limit,)
        )

    rows = cur.fetchall()
    conn.close()

    records = [
        {
            "id": r[0],
            "record_type": r[1],
            "payload": json.loads(r[2]),
            "created_at": r[3],
            "updated_at": r[4]
        }
        for r in rows
    ]

    return {
        "status_code": 200,
        "message": "Search completed",
        "data": {
            "count": len(records),
            "records": records
        },
        "errors": []
    }


def db_update_record(record_id: str, patch: Dict[str, Any]) -> Dict[str, Any]:
    current = db_get_record(record_id)
    if current["status_code"] != 200:
        return current

    payload = current["data"]["payload"]
    payload.update(patch)
    payload["id"] = record_id

    ts = now_iso()
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        UPDATE api_records
        SET payload_json = ?, updated_at = ?
        WHERE id = ?
        """,
        (json.dumps(payload, ensure_ascii=False), ts, record_id)
    )
    conn.commit()
    conn.close()

    return {
        "status_code": 200,
        "message": "Record updated",
        "data": {
            "id": record_id,
            "payload": payload
        },
        "errors": []
    }


def db_delete_record(record_id: str) -> Dict[str, Any]:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("DELETE FROM api_records WHERE id = ?", (record_id,))
    affected = cur.rowcount
    conn.commit()
    conn.close()

    if affected == 0:
        return {
            "status_code": 404,
            "message": "Record not found",
            "data": None,
            "errors": [{"field": "record_id", "message": "No record deleted"}]
        }

    return {
        "status_code": 200,
        "message": "Record deleted",
        "data": {"id": record_id},
        "errors": []
    }


TOOL_REGISTRY = {
    "db_insert_record": db_insert_record,
    "db_get_record": db_get_record,
    "db_search_records": db_search_records,
    "db_update_record": db_update_record,
    "db_delete_record": db_delete_record,
}

## 4) Function declarations لـ Gemini Tool Calling

In [ ]:
FUNCTION_DECLARATIONS = [
    {
        "name": "db_insert_record",
        "description": "Save a structured record to the local database. Use this for create endpoints such as project, task, customer, invoice, report, and workflow.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "record_type": {
                    "type": "STRING",
                    "description": "Record category, e.g. project, task, customer, invoice, report, workflow."
                },
                "payload": {
                    "type": "OBJECT",
                    "description": "Full structured JSON payload to save."
                }
            },
            "required": ["record_type", "payload"]
        }
    },
    {
        "name": "db_get_record",
        "description": "Retrieve one record from the database by its id.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "record_id": {
                    "type": "STRING",
                    "description": "The id of the record."
                }
            },
            "required": ["record_id"]
        }
    },
    {
        "name": "db_search_records",
        "description": "Search records from the database by type and/or text query.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "record_type": {
                    "type": "STRING",
                    "description": "Optional record type filter."
                },
                "query": {
                    "type": "STRING",
                    "description": "Optional text query searched inside JSON payload."
                },
                "limit": {
                    "type": "INTEGER",
                    "description": "Maximum results, from 1 to 100."
                }
            },
            "required": []
        }
    },
    {
        "name": "db_update_record",
        "description": "Patch an existing record by id.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "record_id": {
                    "type": "STRING",
                    "description": "The id of the record to update."
                },
                "patch": {
                    "type": "OBJECT",
                    "description": "Fields to merge into the existing payload."
                }
            },
            "required": ["record_id", "patch"]
        }
    },
    {
        "name": "db_delete_record",
        "description": "Delete a record by id. Use only if the user clearly requests deletion.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "record_id": {
                    "type": "STRING",
                    "description": "The id of the record to delete."
                }
            },
            "required": ["record_id"]
        }
    }
]

## 5) System Prompt: تعريف API كامل داخل البرومت فقط

هنا التعقيد كله موجود في السيستم برومت:
- endpoints
- required fields
- status codes
- workflow rules
- متى يستخدم tool calling
- شكل JSON النهائي

In [ ]:
SYSTEM_PROMPT = """
You are PromptAPI Runtime, an LLM-as-API router and controller.

Core mission:
Convert natural language user requests into structured API-like JSON responses.
You must behave like an API gateway controlled by this system prompt.

Important:
- The endpoints are defined ONLY in this system prompt.
- Never invent endpoints outside the Endpoint Registry.
- Always return a final strict JSON object.
- Use tools only when a database action is required.
- You may call database tools to save, retrieve, search, update, or delete records.
- The backend code is the authority; you are the router and argument extractor.
- If required fields are missing, return 422.
- If intent is unclear, return 400.
- If no endpoint matches, return 404.
- If user asks to delete without a clear record_id, return 422.
- For dangerous or irreversible actions, ask for confirmation in JSON with status_code 409.

Global Response Schema:
{
  "status_code": number,
  "endpoint": string | null,
  "method": string | null,
  "message": string,
  "data": object | null,
  "errors": array,
  "tool_calls": array,
  "meta": {
    "confidence": number,
    "request_type": "single" | "workflow" | "query" | "mutation",
    "timestamp": string
  }
}

Status codes:
- 200 success
- 201 created
- 400 unclear request
- 401 authentication required
- 403 forbidden
- 404 endpoint not found
- 409 confirmation/conflict required
- 422 validation error
- 500 internal error

Endpoint Registry:

1) create_project
Method: POST
Description: Create a complex project with goals, milestones, owner, budget, and risks.
Required:
- project_name: string
- owner: string
- goals: array of strings
Optional:
- deadline: string
- budget: number
- risks: array of strings
Database:
- Must call db_insert_record with record_type="project".
Success:
- Return status_code 201.

2) create_task
Method: POST
Description: Create a task, optionally linked to a project.
Required:
- title: string
- priority: enum(low, medium, high, critical)
Optional:
- project_id: string
- assignee: string
- due_date: string
- dependencies: array of strings
- estimated_hours: number
Database:
- Must call db_insert_record with record_type="task".
Success:
- Return status_code 201.

3) update_task_status
Method: PATCH
Description: Update task status.
Required:
- task_id: string
- status: enum(todo, in_progress, blocked, done, cancelled)
Optional:
- notes: string
Database:
- Must call db_update_record with record_id=task_id.
Success:
- Return status_code 200.

4) get_record
Method: GET
Description: Retrieve any record by id.
Required:
- record_id: string
Database:
- Must call db_get_record.
Success:
- Return status_code from tool result.

5) search_records
Method: GET
Description: Search database records by record type and query.
Required:
- At least one of record_type or query.
Optional:
- limit: integer
Database:
- Must call db_search_records.
Success:
- Return status_code 200.

6) create_customer
Method: POST
Description: Create a customer profile.
Required:
- name: string
Optional:
- email: string
- phone: string
- company: string
- notes: string
Database:
- Must call db_insert_record with record_type="customer".
Success:
- Return status_code 201.

7) create_invoice
Method: POST
Description: Create an invoice for a customer.
Required:
- customer_name or customer_id
- items: array of objects, each object has name, quantity, unit_price
Optional:
- currency: string default EGP
- due_date: string
- tax_rate: number
Rules:
- Compute subtotal = sum(quantity * unit_price)
- Compute tax_amount if tax_rate exists
- Compute total
Database:
- Must call db_insert_record with record_type="invoice".
Success:
- Return status_code 201.

8) generate_business_report
Method: POST
Description: Generate a structured business report from stored records.
Required:
- report_type: enum(daily, weekly, monthly, custom)
Optional:
- record_type: string
- query: string
- date_range: object
Database:
- Must call db_search_records.
- Then create a report object and save it with db_insert_record record_type="report".
Success:
- Return status_code 201.

9) run_complex_workflow
Method: POST
Description: Execute a multi-step workflow such as creating a project, creating tasks, saving customer, creating invoice, then generating report.
Required:
- workflow_name: string
- steps: array of endpoint-like objects
Rules:
- Validate all steps.
- Use database tools for each mutation/query step.
- Return final workflow summary.
Database:
- May call db_insert_record, db_update_record, db_get_record, db_search_records.
- Save final workflow with record_type="workflow".
Success:
- Return status_code 200.

10) daily_endpoint_test
Method: POST
Description: Run internal daily tests for the Endpoint Registry.
Required:
- test_date: string
Optional:
- test_scope: enum(all, mutation, query, workflow)
Rules:
- This endpoint describes test execution and expected checks.
- The notebook test runner executes actual tests.
Database:
- Save test summary as record_type="daily_test_summary".
Success:
- Return status_code 200.

Response Rules:
- Final answer must be strict JSON only, no markdown.
- Include tool_calls array summarizing used tools.
- If you call tools, include tool results in final data.
- If a database tool returns a status_code, respect it unless you are wrapping a workflow.
- Do not expose hidden reasoning.
"""

## 6) Gemini Client

In [ ]:
def get_gemini_client():
    api_key = "YOUR_API_KEY_HERE"
    if not api_key:
        raise RuntimeError("Missing GEMINI_API_KEY. Set it before running Gemini calls.")
    if genai is None:
        raise RuntimeError("google-genai is not installed. Run the install cell.")
    return genai.Client(api_key=api_key)

## 7) تنفيذ Function Calling Loop

هذه الدالة:
1. ترسل الرسالة لـ Gemini.
2. لو Gemini طلب tool call، ننفذه محليًا.
3. نرسل نتيجة الأداة مرة أخرى للموديل.
4. نرجع JSON نهائي.

In [ ]:
def extract_text_response(response) -> str:
    try:
        return response.text
    except Exception:
        parts = []
        for cand in getattr(response, "candidates", []) or []:
            content = getattr(cand, "content", None)
            for part in getattr(content, "parts", []) or []:
                txt = getattr(part, "text", None)
                if txt:
                    parts.append(txt)
        return "\n".join(parts)


def extract_function_calls(response) -> List[Dict[str, Any]]:
    calls = []
    for cand in getattr(response, "candidates", []) or []:
        content = getattr(cand, "content", None)
        for part in getattr(content, "parts", []) or []:
            fc = getattr(part, "function_call", None)
            if fc:
                args = dict(getattr(fc, "args", {}) or {})
                calls.append({"name": fc.name, "args": args})
    return calls


def safe_json_loads(text: str) -> Dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json\n", "", 1).strip()
    try:
        return json.loads(text)
    except Exception:
        return {
            "status_code": 500,
            "endpoint": None,
            "method": None,
            "message": "Model did not return valid JSON",
            "data": {"raw_text": text},
            "errors": [{"message": "Invalid JSON from model"}],
            "tool_calls": [],
            "meta": {"confidence": 0, "request_type": "unknown", "timestamp": now_iso()}
        }


def call_tool(name: str, args: Dict[str, Any]) -> Dict[str, Any]:
    if name not in TOOL_REGISTRY:
        return {
            "status_code": 404,
            "message": f"Tool {name} not found",
            "data": None,
            "errors": [{"message": "Unknown tool"}]
        }
    try:
        return TOOL_REGISTRY[name](**args)
    except Exception as e:
        return {
            "status_code": 500,
            "message": f"Tool {name} failed",
            "data": None,
            "errors": [{"message": str(e)}]
        }


def log_audit(user_message: str, response_obj: Dict[str, Any], request_json: Optional[Dict[str, Any]] = None):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        INSERT INTO audit_logs (id, user_message, endpoint, status_code, request_json, response_json, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (
            new_id("audit"),
            user_message,
            response_obj.get("endpoint"),
            response_obj.get("status_code"),
            json.dumps(request_json or {}, ensure_ascii=False),
            json.dumps(response_obj, ensure_ascii=False),
            now_iso()
        )
    )
    conn.commit()
    conn.close()


def gemini_api_chat(user_message: str, max_tool_rounds: int = 5) -> Dict[str, Any]:
    client = get_gemini_client()

    tools = [
        types.Tool(function_declarations=FUNCTION_DECLARATIONS)
    ]

    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        tools=tools,
        temperature=0.1,
    )

    contents = [
        types.Content(role="user", parts=[types.Part(text=user_message)])
    ]

    used_tool_calls = []

    for _ in range(max_tool_rounds):
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=contents,
            config=config
        )

        function_calls = extract_function_calls(response)

        if not function_calls:
            text = extract_text_response(response)
            final = safe_json_loads(text)
            final.setdefault("tool_calls", used_tool_calls)
            log_audit(user_message, final)
            return final

        # Add model response with function calls to conversation
        try:
            contents.append(response.candidates[0].content)
        except Exception:
            pass

        function_response_parts = []

        for call in function_calls:
            name = call["name"]
            args = call["args"]
            result = call_tool(name, args)

            used_tool_calls.append({
                "name": name,
                "args": args,
                "result_status_code": result.get("status_code")
            })

            function_response_parts.append(
                types.Part.from_function_response(
                    name=name,
                    response={"result": result}
                )
            )

        contents.append(
            types.Content(
                role="tool",
                parts=function_response_parts
            )
        )

    final = {
        "status_code": 500,
        "endpoint": None,
        "method": None,
        "message": "Max tool rounds exceeded",
        "data": None,
        "errors": [{"message": "The model requested too many tool calls"}],
        "tool_calls": used_tool_calls,
        "meta": {"confidence": 0, "request_type": "unknown", "timestamp": now_iso()}
    }
    log_audit(user_message, final)
    return final

## 8) Local Mock Router للاختبارات بدون Gemini

لو مش عايز تستهلك API calls أثناء التطوير، استخدم الدالة دي.  
هي لا تغطي كل الذكاء، لكنها تسمح بتجربة الداتا بيز والاختبارات.

In [ ]:
def local_mock_chat(user_message: str) -> Dict[str, Any]:
    msg = user_message.lower()

    if "project" in msg or "مشروع" in msg:
        result = db_insert_record("project", {
            "project_name": "Demo Project",
            "owner": "Loai",
            "goals": ["Build LLM as API", "Test endpoint registry"],
            "risks": ["Prompt injection", "Invalid JSON"],
            "created_by": "local_mock"
        })
        return {
            "status_code": 201,
            "endpoint": "create_project",
            "method": "POST",
            "message": "Project created by local mock",
            "data": result["data"],
            "errors": [],
            "tool_calls": [{"name": "db_insert_record", "result_status_code": result["status_code"]}],
            "meta": {"confidence": 0.7, "request_type": "mutation", "timestamp": now_iso()}
        }

    if "task" in msg or "تاسك" in msg:
        result = db_insert_record("task", {
            "title": "Demo Task",
            "priority": "high",
            "status": "todo",
            "created_by": "local_mock"
        })
        return {
            "status_code": 201,
            "endpoint": "create_task",
            "method": "POST",
            "message": "Task created by local mock",
            "data": result["data"],
            "errors": [],
            "tool_calls": [{"name": "db_insert_record", "result_status_code": result["status_code"]}],
            "meta": {"confidence": 0.7, "request_type": "mutation", "timestamp": now_iso()}
        }

    if "search" in msg or "ابحث" in msg or "هات" in msg:
        result = db_search_records(limit=10)
        return {
            "status_code": 200,
            "endpoint": "search_records",
            "method": "GET",
            "message": "Search completed by local mock",
            "data": result["data"],
            "errors": [],
            "tool_calls": [{"name": "db_search_records", "result_status_code": result["status_code"]}],
            "meta": {"confidence": 0.6, "request_type": "query", "timestamp": now_iso()}
        }

    return {
        "status_code": 404,
        "endpoint": None,
        "method": None,
        "message": "No endpoint matched in local mock",
        "data": None,
        "errors": [{"message": "Try Gemini router for full behavior"}],
        "tool_calls": [],
        "meta": {"confidence": 0.2, "request_type": "unknown", "timestamp": now_iso()}
    }

## 9) أمثلة تشغيل

In [ ]:
# مثال باستخدام Gemini الحقيقي
# تأكد أن GEMINI_API_KEY موجود
response = gemini_api_chat("اعمل مشروع اسمه PromptAPI، المالك Loai، أهدافه بناء LLM API وحفظ الداتا واختبار endpoints")
pretty(response)

/tmp/ipykernel_3640/1751722684.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat() + "Z"


{
  "status_code": 201,
  "endpoint": "create_project",
  "method": "POST",
  "message": "Project 'PromptAPI' created successfully.",
  "data": {
    "id": "project_e29630873ebb",
    "project_name": "PromptAPI",
    "owner": "Loai",
    "goals": [
      "بناء LLM API",
      "حفظ الداتا",
      "اختبار endpoints"
    ]
  },
  "errors": [],
  "tool_calls": [
    {
      "tool_name": "db_insert_record",
      "parameters": {
        "record_type": "project",
        "payload": {
          "project_name": "PromptAPI",
          "owner": "Loai",
          "goals": [
            "بناء LLM API",
            "حفظ الداتا",
            "اختبار endpoints"
          ]
        }
      },
      "result": {
        "data": {
          "id": "project_e29630873ebb",
          "payload": {
            "goals": [
              "بناء LLM API",
              "حفظ الداتا",
              "اختبار endpoints"
            ],
            "id": "project_e29630873ebb",
            "owner": "Loai",
            "pr

In [ ]:
# مثال محلي بدون Gemini
response = local_mock_chat("اعمل مشروع جديد")
pretty(response)

{
  "status_code": 201,
  "endpoint": "create_project",
  "method": "POST",
  "message": "Project created by local mock",
  "data": {
    "id": "project_e31b399583d0",
    "record_type": "project",
    "payload": {
      "project_name": "Demo Project",
      "owner": "Loai",
      "goals": [
        "Build LLM as API",
        "Test endpoint registry"
      ],
      "risks": [
        "Prompt injection",
        "Invalid JSON"
      ],
      "created_by": "local_mock",
      "id": "project_e31b399583d0"
    }
  },
  "errors": [],
  "tool_calls": [
    {
      "name": "db_insert_record",
      "result_status_code": 201
    }
  ],
  "meta": {
    "confidence": 0.7,
    "request_type": "mutation",
    "timestamp": "2026-05-28T00:09:56.512155Z"
  }
}


/tmp/ipykernel_3640/1751722684.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat() + "Z"


In [ ]:
response = local_mock_chat("هات كل الداتا")
pretty(response)

{
  "status_code": 200,
  "endpoint": "search_records",
  "method": "GET",
  "message": "Search completed by local mock",
  "data": {
    "count": 2,
    "records": [
      {
        "id": "project_e31b399583d0",
        "record_type": "project",
        "payload": {
          "project_name": "Demo Project",
          "owner": "Loai",
          "goals": [
            "Build LLM as API",
            "Test endpoint registry"
          ],
          "risks": [
            "Prompt injection",
            "Invalid JSON"
          ],
          "created_by": "local_mock",
          "id": "project_e31b399583d0"
        },
        "created_at": "2026-05-28T00:09:56.501039Z",
        "updated_at": "2026-05-28T00:09:56.501039Z"
      },
      {
        "id": "project_e29630873ebb",
        "record_type": "project",
        "payload": {
          "goals": [
            "بناء LLM API",
            "حفظ الداتا",
            "اختبار endpoints"
          ],
          "owner": "Loai",
          "project

/tmp/ipykernel_3640/1751722684.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat() + "Z"


## 10) Daily Endpoint Tests داخل النوتبوك

الاختبارات هنا تعمل على النظام نفسه:
- إنشاء project
- إنشاء task
- البحث في records
- تحديث record
- إنشاء invoice / workflow عبر Gemini لو متاح
- حفظ نتائج الاختبار في جدول `daily_tests`

تقدر تشغلها يوميًا يدويًا، أو تربط النوتبوك بـ cron/Colab scheduler.

In [ ]:
class TestResult(BaseModel):
    test_name: str
    passed: bool
    endpoint: Optional[str] = None
    response: Dict[str, Any]


def save_test_result(result: TestResult):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        INSERT INTO daily_tests (id, test_name, passed, endpoint, response_json, created_at)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            new_id("test"),
            result.test_name,
            1 if result.passed else 0,
            result.endpoint,
            json.dumps(result.response, ensure_ascii=False),
            now_iso()
        )
    )
    conn.commit()
    conn.close()


def assert_status(response: Dict[str, Any], allowed: List[int]) -> bool:
    return int(response.get("status_code", -1)) in allowed


def run_daily_tests(use_gemini: bool = False) -> Dict[str, Any]:
    router = gemini_api_chat if use_gemini else local_mock_chat

    tests = [
        {
            "name": "create_project_basic",
            "prompt": "اعمل مشروع اسمه AI Sales Platform، المالك Loai، الأهداف تحليل المبيعات وبناء تقارير يومية",
            "allowed": [200, 201],
            "expected_endpoint": "create_project"
        },
        {
            "name": "create_task_basic",
            "prompt": "اعمل تاسك مهم اسمه راجع تصميم قاعدة البيانات وخليه high priority",
            "allowed": [200, 201],
            "expected_endpoint": "create_task"
        },
        {
            "name": "search_records_basic",
            "prompt": "هات كل الريكوردز اللي عندك في الداتا بيز",
            "allowed": [200],
            "expected_endpoint": "search_records"
        },
    ]

    results = []

    for t in tests:
        try:
            response = router(t["prompt"])
            passed = assert_status(response, t["allowed"])

            # في mock ممكن endpoint يختلف حسب البساطة، وفي Gemini نتوقع دقة أعلى
            if use_gemini and t.get("expected_endpoint"):
                passed = passed and response.get("endpoint") == t["expected_endpoint"]

            tr = TestResult(
                test_name=t["name"],
                passed=passed,
                endpoint=response.get("endpoint"),
                response=response
            )
        except Exception as e:
            tr = TestResult(
                test_name=t["name"],
                passed=False,
                endpoint=None,
                response={
                    "status_code": 500,
                    "message": str(e),
                    "errors": [{"message": str(e)}]
                }
            )

        save_test_result(tr)
        results.append(tr.model_dump())

    summary = {
        "status_code": 200,
        "endpoint": "daily_endpoint_test",
        "method": "POST",
        "message": "Daily tests completed",
        "data": {
            "test_date": str(date.today()),
            "total": len(results),
            "passed": sum(1 for r in results if r["passed"]),
            "failed": sum(1 for r in results if not r["passed"]),
            "results": results
        },
        "errors": [],
        "tool_calls": [{"name": "daily_tests_table_insert", "result_status_code": 201}],
        "meta": {"confidence": 1.0, "request_type": "workflow", "timestamp": now_iso()}
    }

    db_insert_record("daily_test_summary", summary["data"])
    return summary

In [ ]:
# اختبارات يومية بدون Gemini
daily_summary = run_daily_tests(use_gemini=False)
pretty(daily_summary)

{
  "status_code": 200,
  "endpoint": "daily_endpoint_test",
  "method": "POST",
  "message": "Daily tests completed",
  "data": {
    "test_date": "2026-05-28",
    "total": 3,
    "passed": 3,
    "failed": 0,
    "results": [
      {
        "test_name": "create_project_basic",
        "passed": true,
        "endpoint": "create_project",
        "response": {
          "status_code": 201,
          "endpoint": "create_project",
          "method": "POST",
          "message": "Project created by local mock",
          "data": {
            "id": "project_2ca0b30a1e8e",
            "record_type": "project",
            "payload": {
              "project_name": "Demo Project",
              "owner": "Loai",
              "goals": [
                "Build LLM as API",
                "Test endpoint registry"
              ],
              "risks": [
                "Prompt injection",
                "Invalid JSON"
              ],
              "created_by": "local_mock",
          

/tmp/ipykernel_3640/1751722684.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat() + "Z"


In [ ]:
# اختبارات يومية باستخدام Gemini الحقيقي
daily_summary_gemini = run_daily_tests(use_gemini=True)
pretty(daily_summary_gemini)

/tmp/ipykernel_3640/1751722684.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat() + "Z"


{
  "status_code": 200,
  "endpoint": "daily_endpoint_test",
  "method": "POST",
  "message": "Daily tests completed",
  "data": {
    "test_date": "2026-05-28",
    "total": 3,
    "passed": 3,
    "failed": 0,
    "results": [
      {
        "test_name": "create_project_basic",
        "passed": true,
        "endpoint": "create_project",
        "response": {
          "status_code": 201,
          "endpoint": "create_project",
          "method": "POST",
          "message": "Project 'AI Sales Platform' created successfully.",
          "data": {
            "id": "project_117b4d89cc6a",
            "payload": {
              "goals": [
                "تحليل المبيعات",
                "بناء تقارير يومية"
              ],
              "id": "project_117b4d89cc6a",
              "owner": "Loai",
              "project_name": "AI Sales Platform"
            },
            "record_type": "project"
          },
          "errors": [],
          "tool_calls": [
            {
         

## 11) قراءة نتائج الاختبارات اليومية من قاعدة البيانات

In [ ]:
def get_daily_test_results(limit: int = 20):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        SELECT test_name, passed, endpoint, response_json, created_at
        FROM daily_tests
        ORDER BY created_at DESC
        LIMIT ?
        """,
        (limit,)
    )
    rows = cur.fetchall()
    conn.close()

    return [
        {
            "test_name": r[0],
            "passed": bool(r[1]),
            "endpoint": r[2],
            "response": json.loads(r[3]),
            "created_at": r[4]
        }
        for r in rows
    ]

pretty(get_daily_test_results())

## 12) تطوير لاحق

أفكار جاهزة للإضافة:
- استبدال SQLite بـ PostgreSQL أو Supabase.
- إضافة Auth و permissions لكل endpoint.
- إضافة JSON Schema validation خارج الموديل.
- إضافة endpoint versioning.
- إضافة replay لاختبارات يومية.
- إضافة FastAPI حول `gemini_api_chat`.
- إضافة dashboard لعرض audit logs وdaily tests.